# OECD FDI Income: Normality Analysis

This notebook evaluates the numeric variables by keeping the project variable definitions fixed.

- `TIME_PERIOD` is treated as a discrete temporal variable.
- `OBS_VALUE_NUM` is treated as the main continuous quantitative variable.
- Numeric-looking metadata codes are not treated as analytical numeric variables.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

ALPHA = 0.05


C:\Users\Yusuf\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent

data_path = project_root / 'data' / 'processed' / 'OECD_FDI_Except_Pointless.csv'
df = pd.read_csv(data_path, sep=';', encoding='latin1', engine='python', on_bad_lines='skip')

df['OBS_VALUE_NUM'] = pd.to_numeric(
    df['OBS_VALUE'].astype(str).str.replace('.', '', regex=False),
    errors='coerce'
)

display(Markdown(f'**Dataset shape:** {df.shape[0]:,} rows and {df.shape[1]:,} columns'))
df[['TIME_PERIOD', 'OBS_VALUE', 'OBS_VALUE_NUM']].head()


**Dataset shape:** 10,344 rows and 20 columns

,TIME_PERIOD,OBS_VALUE,OBS_VALUE_NUM
0,2024,NaN,NaN
1,2023,19.400.704.272.141,1.940070e+13
2,2023,20.018.603.415.055,2.001860e+13
3,2024,97.339.390.006.489,9.733939e+13
4,2023,NaN,NaN


## Numeric Variables Included in the Analysis

The notebook follows the fixed report-based variable-role logic.


In [3]:
numeric_variables = {
    'TIME_PERIOD': 'Discrete temporal variable',
    'OBS_VALUE_NUM': 'Continuous quantitative variable',
}

pd.DataFrame(
    [{'variable': k, 'variable_type': v} for k, v in numeric_variables.items()]
)


,variable,variable_type
0,TIME_PERIOD,Discrete temporal variable
1,OBS_VALUE_NUM,Continuous quantitative variable


In [4]:
def calculate_summary(series):
    clean = series.dropna()
    mode_values = clean.mode()
    mode_value = mode_values.iloc[0] if not mode_values.empty else np.nan
    return {
        'n': int(clean.shape[0]),
        'missing': int(series.isna().sum()),
        'min': clean.min(),
        'q1': clean.quantile(0.25),
        'median': clean.median(),
        'q3': clean.quantile(0.75),
        'max': clean.max(),
        'mode': mode_value,
        'mean': clean.mean(),
        'variance': clean.var(ddof=1),
        'skewness': clean.skew(),
        'kurtosis': clean.kurt(),
        'unique_values': int(clean.nunique()),
    }


def shapiro_result(series):
    clean = series.dropna()
    sample = clean.sample(n=min(5000, len(clean)), random_state=42) if len(clean) > 5000 else clean
    statistic, p_value = stats.shapiro(sample)
    return float(statistic), float(p_value), int(sample.shape[0])


def dagostino_result(series):
    clean = series.dropna()
    statistic, p_value = stats.normaltest(clean)
    return float(statistic), float(p_value)


def jarque_bera_result(series):
    clean = series.dropna()
    statistic, p_value = stats.jarque_bera(clean)
    return float(statistic), float(p_value)


def evaluate_normality(summary, tests, variable_name):
    rejected_tests = [name for name, result in tests.items() if result['p_value'] < ALPHA]
    abs_skew = abs(summary['skewness'])
    abs_kurt = abs(summary['kurtosis'])
    n = summary['n']

    if variable_name == 'TIME_PERIOD':
        return (
            'Not normal. This variable is a discrete two-year time indicator rather than a continuous '
            'measurement, so a normal distribution is not expected or substantively appropriate.'
        )

    if rejected_tests:
        base_comment = (
            f"Not normal at alpha={ALPHA:.2f}. The following tests reject normality: {', '.join(rejected_tests)}."
        )
    else:
        base_comment = f"Approximately normal at alpha={ALPHA:.2f}; none of the selected tests reject normality."

    if n >= 30 and abs_skew < 2 and abs_kurt < 7:
        clt_comment = (
            ' Although the distribution is not perfectly normal, the deviation is not severe enough to block '
            'parametric analysis in larger samples. We may continue under an approximate normality assumption '
            'by relying on the Central Limit Theorem.'
        )
    else:
        clt_comment = (
            ' The deviation from normality is substantial, so relying on a normality assumption through the '
            'Central Limit Theorem should be done cautiously or supported with robust methods.'
        )

    return base_comment + clt_comment


## Descriptive Statistics

The table below reports minimum, maximum, quartiles, mode, median, mean, variance, kurtosis, and skewness for each numeric variable.


In [5]:
summary_rows = []
for variable, variable_type in numeric_variables.items():
    summary = calculate_summary(df[variable])
    summary_rows.append({'variable': variable, 'variable_type': variable_type, **summary})

summary_df = pd.DataFrame(summary_rows)
summary_df.round(4)


,variable,variable_type,n,missing,min,q1,median,q3,max,mode,mean,variance,skewness,kurtosis,unique_values
0,TIME_PERIOD,Discrete temporal variable,10344,0,2.023000e+03,2023.0,2023.0,2.024000e+03,2.024000e+03,2023.0,2.023467e+03,2.490000e-01,0.1302,-1.9834,2
1,OBS_VALUE_NUM,Continuous quantitative variable,7407,2937,-9.593532e+13,0.0,91139999.0,2.650251e+13,9.984813e+13,0.0,1.590132e+13,7.792772e+26,0.8267,1.4640,2798


### Comment

`TIME_PERIOD` has only two observed values, so its center and spread reflect a coding structure rather than a continuous distribution. `OBS_VALUE_NUM` has a very wide range, a median much lower than the mean, and noticeable positive skewness, which already suggests that perfect normality is unlikely.


## Normality Tests

The notebook applies Shapiro-Wilk, D'Agostino K², and Jarque-Bera tests when sample size conditions allow.


In [6]:
test_rows = []
interpretation_rows = []

for variable, variable_type in numeric_variables.items():
    series = df[variable]
    clean = series.dropna()
    summary = calculate_summary(series)
    tests = {}

    if clean.shape[0] >= 8:
        shapiro_stat, shapiro_p, shapiro_n = shapiro_result(series)
        tests['Shapiro-Wilk'] = {
            'statistic': shapiro_stat,
            'p_value': shapiro_p,
            'sample_size_used': shapiro_n,
        }

    if clean.shape[0] >= 20:
        dag_stat, dag_p = dagostino_result(series)
        tests["D'Agostino K^2"] = {
            'statistic': dag_stat,
            'p_value': dag_p,
            'sample_size_used': int(clean.shape[0]),
        }

        jb_stat, jb_p = jarque_bera_result(series)
        tests['Jarque-Bera'] = {
            'statistic': jb_stat,
            'p_value': jb_p,
            'sample_size_used': int(clean.shape[0]),
        }

    for test_name, result in tests.items():
        test_rows.append({
            'variable': variable,
            'variable_type': variable_type,
            'test': test_name,
            **result,
        })

    interpretation_rows.append({
        'variable': variable,
        'variable_type': variable_type,
        'interpretation': evaluate_normality(summary, tests, variable),
    })

tests_df = pd.DataFrame(test_rows)
interpretation_df = pd.DataFrame(interpretation_rows)

display(tests_df.round(4))
display(interpretation_df)


,variable,variable_type,test,statistic,p_value,sample_size_used
0,TIME_PERIOD,Discrete temporal variable,Shapiro-Wilk,0.6354,0.0,5000
1,TIME_PERIOD,Discrete temporal variable,D'Agostino K^2,35944.4184,0.0,10344
2,TIME_PERIOD,Discrete temporal variable,Jarque-Bera,1724.1239,0.0,10344
3,OBS_VALUE_NUM,Continuous quantitative variable,Shapiro-Wilk,0.8307,0.0,5000
4,OBS_VALUE_NUM,Continuous quantitative variable,D'Agostino K^2,892.4244,0.0,7407
5,OBS_VALUE_NUM,Continuous quantitative variable,Jarque-Bera,1503.2164,0.0,7407


,variable,variable_type,interpretation
0,TIME_PERIOD,Discrete temporal variable,Not normal. This variable is a discrete two-ye...
1,OBS_VALUE_NUM,Continuous quantitative variable,Not normal at alpha=0.05. The following tests ...


### Comment

`TIME_PERIOD` is not normal, but this result is expected because it is a discrete year indicator with only two values. `OBS_VALUE_NUM` is also rejected by all selected normality tests, so the variable is not normally distributed in a strict statistical sense.


## Final Interpretation

For `OBS_VALUE_NUM`, the normality tests reject exact normality. However, the sample is large and the skewness-kurtosis profile is not extreme enough to make parametric continuation automatically unreasonable. Therefore, if the next analysis step does not show a severe additional violation, we may proceed under an approximate normality assumption by relying on the Central Limit Theorem.
